In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import optuna
import os
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 결측치 처리
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# 2. 파생변수 및 봉우리(Spike) 타겟팅 강력한 교호작용(Interaction) 변수 추가
def add_features(df):
    data = df.copy()
    
    # 기본 파생변수
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    
    # [신규] 핵심 교호작용(Interaction) 변수
    data['is_diabetic_and_overworking'] = ((data['medical_history'] == 'diabetes') & (data['is_overworking'] == 1)).astype(int)
    data['is_heart_disease_low_score'] = ((data['medical_history'] == 'heart disease') & (data['sleep_pattern'] != 'sleep difficulty')).astype(int)
    data['is_oversleeping_extreme'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    
    return data

train = add_features(train)
test = add_features(test)

# 3. 이상치 처리 (IQR Clipping)
top_num_features = ['cholesterol', 'height', 'glucose', 'glucose_chol_ratio', 'bmi', 'weight', 'cardio_metabolic_load', 'map']
for col in top_num_features:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    train[col] = train[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    test[col] = test[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# 4. Target Encoding (K-Fold 방식을 사용하여 Data Leakage 방지)
target_encode_cols = ['medical_history', 'sleep_pattern', 'family_medical_history']
kf_te = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_encode_cols:
    train[f'{col}_te'] = np.nan
    
    # Train 데이터 타겟 인코딩
    for tr_idx, val_idx in kf_te.split(train):
        X_tr, X_val = train.iloc[tr_idx], train.iloc[val_idx]
        target_mean = X_tr.groupby(col)['stress_score'].mean()
        train.loc[val_idx, f'{col}_te'] = X_val[col].map(target_mean)
    
    # Test 데이터는 Train 전체의 평균으로 인코딩
    test[f'{col}_te'] = test[col].map(train.groupby(col)['stress_score'].mean())
    
    # 결측치(새로운 카테고리 등)는 전체 평균으로 대체
    train[f'{col}_te'] = train[f'{col}_te'].fillna(train['stress_score'].mean())
    test[f'{col}_te'] = test[f'{col}_te'].fillna(train['stress_score'].mean())
    
    # 인코딩이 완료된 기존 범주형 컬럼 삭제
    train = train.drop(columns=[col])
    test = test.drop(columns=[col])

# 남은 범주형 변수 처리
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

cat_cols = ['gender', 'smoke_status', 'activity', 'edu_level']
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score'] # [신규] 로그 변환(log1p) 영구 폐기 및 스케일 복구
x_test = test.drop('ID', axis=1)

# 5. Optuna 하이퍼파라미터 튜닝
def objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255, step=16),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_mae = []
    
    for tr_idx, val_idx in kf.split(x_train):
        X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, categorical_feature=cat_cols)
        
        pred = model.predict(X_val) # 역변환 삭제
        cv_mae.append(mean_absolute_error(y_val, pred))
        
    return np.mean(cv_mae)

print("★ Optuna 튜닝 시작 (30회 반복) ★")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print(f"\n★ Best Optuna CV MAE: {study.best_value:.4f}")

# 6. 최적 파라미터로 최종 학습 및 예측
best_params = study.best_params
best_params.update({'objective': 'regression_l1', 'metric': 'mae', 'random_state': 42, 'verbose': -1})

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))

for tr_idx, val_idx in kf.split(x_train):
    X_tr, X_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_tr, y_tr, categorical_feature=cat_cols)
    
    oof_preds[val_idx] = final_model.predict(X_val)
    test_preds += final_model.predict(x_test) / kf.n_splits

final_cv_mae = mean_absolute_error(y_train, oof_preds)
print(f"\n★ 최종 모델 자체 점수 (OOF CV MAE): {final_cv_mae:.4f}")

# 7. 예측값 클리핑 및 제출 파일 저장
test_preds = np.clip(test_preds, 0, 1)
sample_submission['stress_score'] = test_preds
submit_path = '../submissions/submit_14_magic_features_te.csv'
os.makedirs('../submissions', exist_ok=True)
sample_submission.to_csv(submit_path, index=False)
print(f"★ 제출 파일 생성 완료: {submit_path}")